<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day1/D1_RAG_Assistant_EN_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>
 <span style="color:#6B7B75;">Day 1 · open track</span>
 <span style="flex:1;"></span>
 <a href="./D1_RAG_Assistant_FR_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 Français</a>
 <a href="./D1_RAG_Assistant_EN.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ guided track</a>
</div>

<!-- Build a RAG Assistant Over Your Own Publications · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d1_rag_assistant.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;margin-bottom:6px;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">African Development Bank · AU STATAFRIC · STG17 · Day 1 · 14:45</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  A RAG assistant over your own publications</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  By the end of this laboratory you will have an assistant that answers questions about your
  office's documents, <b>cites the passage it used</b>, and says so when the answer is not there.
  That last behaviour is the one that makes it publishable.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  90 minutes · no GPU · works on a laptop, on Colab and on Kaggle</div>
</div>

> **What you are building, in one sentence.** Not a model that knows your
> statistics — this morning established that no model does. A pipeline that
> *retrieves* the right paragraph from your own documents and asks a model to
> answer **from that paragraph only**.


### The road through this laboratory

| # | Step | What you learn |
|---|------|----------------|
| 1 | Setup and a model provider | Which paths work when a key is missing |
| 2 | Load a corpus | Why a scanned PDF is invisible to your assistant |
| 3 | Cut it into passages | What chunk size and overlap actually change |
| 4 | Two retrievers, compared | Word matching versus meaning matching, and how each fails |
| 5 | The prompt that forbids invention | Where the refusal rule lives |
| 6 | Ask, and read the citation | An answer without its source is not usable |
| 7 | The refusal test | A system that always answers is broken |
| 8 | Calibrate the score floor | Turning "always answers" into "answers when it can" |
| 9 | Evaluate and save | Separating a retrieval problem from a generation problem |

**Requirements.** `scikit-learn`, `pandas`, and the `stg17` toolkit. Optional:
`sentence-transformers` for the meaning-based retriever (about 90 MB, once), and
`pypdf` if your corpus is PDF. The next cell installs what is missing.

**A model provider is optional for steps 1–4 and 8–9.** Steps 5–7 need one. If
you have no API key, the notebook tells you which paths remain open.


In [ ]:
# The workshop toolkit, plus what this laboratory needs. Safe to re-run.
import subprocess
import sys

REPO = "https://github.com/STG17-Africa/stg17-workshop"

try:
    import stg17  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}.git"], check=False)

from stg17 import env, i18n, rag, ui  # noqa: E402
from stg17.i18n import T  # noqa: E402

S = env.setup({"scikit-learn": "scikit-learn>=1.3", "pandas": "pandas>=2.0"}, lang="EN")
print(S.summary())

---
## 1 · One variable, and a model provider

`COUNTRY_ISO3` decides where your outputs are written and how they are labelled.
Change it to your own country and the rest of the notebook follows — the registry
covers all 55 African Union member states.

The provider table below is the honest picture of what you can run today.


In [ ]:
# Change this one line to run the whole notebook on your own country.
COUNTRY_ISO3 = "CIV"

from stg17 import countries, llm  # noqa: E402

C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d1_rag")
print(f"{C.name_en} / {C.name_fr}  ({C.iso3})  ->  {OUT}")

ui.status_table(
    llm.status(),
    title=T("Model providers on this machine", "Fournisseurs de modèles sur cette machine"),
    note=T("You need exactly one. If all are unavailable, steps 1-4 and 8-9 still run: "
           "they are the retrieval half, and retrieval is where most RAG problems live.",
           "Un seul suffit. Si aucun n'est disponible, les étapes 1 à 4 et 8 à 9 "
           "fonctionnent quand même : c'est la moitié « récupération », et c'est là que "
           "vivent la plupart des problèmes du RAG."),
)

---
## 2 · The corpus

Two ways to proceed, and the second is not a consolation prize:

**Your own documents.** Put PDFs, Word exports or text files in a folder and set
`CORPUS_DIR` to it. This is the real exercise — your assistant is only as good as
what you feed it.

**The sample corpus.** Five short fictional publications in the shapes an office
actually produces: a census report, a methodological note, a survey bulletin, a
summary of confidentiality provisions, a dissemination calendar. Every figure in
them is invented and every file says so on its first line.

> **Why the sample is deliberately fictional.** A convincing sample corpus is the
> fastest route from a teaching exercise to an invented number in a real
> publication. If yours reads like your own statistics, someone will eventually
> quote it.


In [ ]:
# Set this to a folder of your own documents, or leave it None for the sample.
CORPUS_DIR = None

if CORPUS_DIR is None:
    CORPUS_DIR = rag.write_sample_corpus(S.path("corpus", "sample"))
    print(T(f"Using the fictional sample corpus at {CORPUS_DIR}",
            f"Corpus d'exemple fictif utilisé : {CORPUS_DIR}"))

docs = rag.load_corpus(CORPUS_DIR)

import pandas as pd  # noqa: E402

ui.result_table(
    pd.DataFrame([{"title": d.title, "words": d.words, "source": d.source} for d in docs]),
    caption=T("Every document your assistant can see. A file missing here is a file "
              "your assistant will never mention — and it will not tell you that.",
              "Tous les documents que votre assistant peut voir. Un fichier absent ici "
              "est un fichier que votre assistant ne mentionnera jamais — et il ne vous "
              "le dira pas."),
)

### The failure that is hardest to diagnose

A scanned PDF is an image. It has no text layer, `load_corpus` extracts nothing
from it, and it is reported and skipped. Without that report, the symptom is an
assistant that calmly does not know about a report sitting in its folder.

If your own documents are scans, they need OCR before this laboratory can use
them — and that is a Day 3 conversation, not a Day 1 one.


---
## 3 · Cutting documents into passages

The model is never given a whole document. It is given a few passages, so the
passage boundary decides what it can see at once.

Two numbers control this:

- **`size`** — how long a passage is, in characters. Too small and the answer is
  split across two passages, only one of which is retrieved. Too large and the
  passage carries three topics, so the retriever cannot tell which one you asked
  about.
- **`overlap`** — how much of the previous passage is repeated at the start of the
  next. It exists because the sentence that answers a question is often the one
  that straddles a boundary.

Run the cell, then **change the numbers and run it again**. Watch the passage
count and read one passage each time. There is no correct value — there is a
value that suits your documents, and you find it by looking.


In [ ]:
# TODO: Chunk the documents with rag.chunk_documents, then print the count and one passage
...

---
## 4 · Two retrievers, and how each one fails

This is the step that repays the most attention.

**TF-IDF** matches words. It needs no download and works on any network. Ask it
about "population" and it finds paragraphs containing "population". Ask it "how
many people live there" and it may find nothing at all, because those words do
not appear.

**Embeddings** match meaning. A model turns each passage into a vector, and
"how many people live there" lands near a paragraph about population even with no
shared word. It costs a one-off download of about 90 MB.

They fail in opposite ways, and the difference matters:

- TF-IDF returns **nothing** when the words do not match. Honest, and easy to detect.
- Embeddings return **the nearest passage**, always — even when nothing in your
  corpus is close. Confident, and invisible without a score threshold.

The cell below builds both and asks them the same questions.


In [ ]:
# TF-IDF always works. Embeddings are attempted and reported honestly if unavailable.
tfidf = rag.build_index(chunks, kind="tfidf")

try:
    S.installed += env.ensure({"sentence-transformers": "sentence-transformers>=2.2"})
    dense = rag.build_index(chunks, kind="embedding")
except Exception as exc:  # noqa: BLE001
    dense = None
    print(T(f"Embedding retriever unavailable ({type(exc).__name__}). The laboratory "
            f"continues on TF-IDF; the comparison below will show one column only.",
            f"Récupérateur sémantique indisponible ({type(exc).__name__}). Le laboratoire "
            f"continue en TF-IDF ; la comparaison ci-dessous n'aura qu'une colonne."))

In [ ]:
# TODO: For each probe question, print the top passage from each retriever side by side
...

> **Which should you use?** Embeddings, where you can. But an office on a
> constrained network running TF-IDF with a well-written question set gets a
> working assistant, and a working assistant beats a planned one. Note in your
> documentation which retriever produced your results — they are not
> interchangeable, and a reader cannot tell from the output.


---
## 5 · The prompt that forbids invention

Retrieval is half the system. The other half is an instruction strict enough that
the model does not fall back on its own memory when the passages are unhelpful.

The rule that does the work is the first one: when the passages do not contain
the answer, reply with a fixed refusal string and nothing else. A fixed string,
not a polite sentence, because the notebook needs to *detect* the refusal in
order to count it.


In [ ]:
# Read this. It is short, and every line of it is load-bearing.
print(rag.SYSTEM)
print("\n" + "=" * 70)
print(T("The refusal string the notebook looks for:", "La chaîne de refus recherchée :"),
      rag.REFUSAL)

In [ ]:
# What the model actually receives — the passages first, the question last.
index = dense if dense is not None else tfidf
preview = rag.build_prompt(PROBES[0], index.search(PROBES[0], k=2))
print(preview[:900] + ("\n...[truncated]" if len(preview) > 900 else ""))

---
## 6 · Ask, and read the citation

Now the three steps run together. Read the answer, then read the passages beneath
it and check that the answer is actually in them.

Do that check by hand at least three times before you trust the pipeline. It is
the only way to develop a feel for when it is working.


In [ ]:
# TODO: Call rag.answer() and display the answer with its passages
...

---
## 7 · The refusal test

This is the step most first builds skip, and it is the one that decides whether
the assistant is publishable.

Ask questions your corpus **cannot** answer. A correct system refuses. An
incorrect one produces a fluent, plausible, unsourced answer — and it will do
that in front of a journalist just as readily as it does here.


In [ ]:
# Questions with no answer in the corpus. Every one should be refused.
UNANSWERABLE = [
    T("What is the current price of maize per kilogram?",
      "Quel est le prix actuel du maïs au kilogramme ?"),
    T("How many hospitals are there in the northern region?",
      "Combien d'hôpitaux compte la région du nord ?"),
    T("What will the population be in 2050?",
      "Quelle sera la population en 2050 ?"),
]

checks = []
if result is not None:
    for question in UNANSWERABLE:
        a = rag.answer(question, index, k=3)
        checks.append({"question": question,
                       "refused": a.refused,
                       "reply": a.text[:90].replace("\n", " ")})
    ui.result_table(
        pd.DataFrame(checks),
        caption=T("Every row should say refused = True. A False is not a small defect: "
                  "it is the system inventing an answer with a source line attached.",
                  "Chaque ligne devrait indiquer refused = True. Un False n'est pas un "
                  "défaut mineur : c'est le système qui invente une réponse en y "
                  "attachant une ligne de source."),
    )
else:
    print(T("Skipped — needs a model provider.", "Ignoré — exige un fournisseur de modèle."))

> **If a row says False.** Do not conclude the model is bad. Look at the passages
> it was given: with an embedding retriever it received the *nearest* passage
> regardless of how far away it was, and something vaguely related is much harder
> to refuse than something obviously irrelevant. That is what step 8 fixes.


---
## 8 · Calibrate the score floor

Every retrieved passage carries a similarity score. Questions your corpus answers
score higher than questions it does not — but the two ranges overlap, and where
they separate depends on your documents.

Find the floor by measuring, not by guessing: score a set of answerable questions
and a set of unanswerable ones, and put the threshold between the distributions.

Too low and irrelevant passages reach the model, which then struggles to refuse.
Too high and real questions get refused. There is no universal value.


In [ ]:
# TODO: Score the answerable and unanswerable questions, and compare their top scores
...

---
## 9 · Evaluate, then save the evidence

Two different things can go wrong, and from the answer alone they look identical:

- **Retrieval failed** — the right paragraph never reached the model. No prompt
  fixes this. Change the chunking, the retriever, or the question.
- **Generation failed** — the right paragraph reached the model and the answer is
  still wrong. Now the prompt is the place to work.

`evaluate_retrieval` measures the first, so you know which half to fix.


In [ ]:
# Which document SHOULD answer each question? That is the whole evaluation set.
CASES = [
    {"question": ANSWERABLE[0], "expect": "labour"},
    {"question": ANSWERABLE[1], "expect": "methodology"},
    {"question": ANSWERABLE[2], "expect": "dissemination"},
    {"question": UNANSWERABLE[0], "expect": None},
]

report = rag.evaluate_retrieval(index, CASES, k=3)
ui.result_table(report, caption=T(
    "hit = the expected document reached the model. A False here means no prompt "
    "change will help.",
    "hit = le document attendu a atteint le modèle. Un False ici signifie qu'aucune "
    "modification du prompt n'y changera rien."))

rate = report["hit"].mean()
print(T(f"Retrieval hit rate: {rate:.0%} on {len(report)} case(s).",
        f"Taux de récupération correcte : {rate:.0%} sur {len(report)} cas."))

In [ ]:
# The deliverable: answers, their passages and their provenance, on disk.
transcript = []
if result is not None:
    for question in ANSWERABLE:
        transcript.append(rag.answer(question, index, k=3, min_score=MIN_SCORE))

path = rag.save_transcript(
    transcript,
    OUT / "rag_transcript.json",
    meta={
        "country": C.iso3,
        "corpus": str(CORPUS_DIR),
        "documents": len(docs),
        "passages": len(chunks),
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "retriever": index.kind,
        "min_score": MIN_SCORE,
        "retrieval_hit_rate": round(float(rate), 3),
    },
)
report.to_csv(OUT / "rag_retrieval_eval.csv", index=False)
print(T(f"Written: {path.name} and rag_retrieval_eval.csv in {OUT}",
        f"Écrits : {path.name} et rag_retrieval_eval.csv dans {OUT}"))

---
### What you have

<table>
<tr><td><b>rag_transcript.json</b></td><td>Every answer with the exact passages behind it,
the retriever used, the chunk parameters and the score floor.</td></tr>
<tr><td><b>rag_retrieval_eval.csv</b></td><td>Which questions retrieved the right document,
and which did not.</td></tr>
</table>

Both go into your country repository on Friday. An answer without the passage it
came from cannot be checked by anyone else, and an assistant whose outputs nobody
can audit is not one a statistical office can stand behind.

### The limits of what you built

- It answers from **your documents only**. A question about anything else is
  correctly refused, and that is a feature.
- Retrieval quality is bounded by chunking. A figure split across two passages
  may never be retrieved whole.
- The score floor was calibrated on a handful of questions. Twenty would be
  better; a hundred would be defensible.
- **Nothing here validates the answer against the source.** It cites a passage;
  it does not prove the answer is in it. A human still reads before publication.

### Checkpoint

| Question | Answer |
|---|---|
| Where does the refusal rule live? | In the system prompt — rule 1 |
| A wrong answer with the right passage attached: what do you fix? | The prompt |
| A wrong answer with an irrelevant passage: what do you fix? | The retrieval — chunking, retriever, or score floor |
| Why is "not in my documents" a good answer? | Because the alternative is a fluent invention with a citation attached |


---
## Your turn

Four extensions, in increasing order of difficulty. The first two are worth doing
before Friday.

1. **Use your own documents.** Point `CORPUS_DIR` at real publications from your
   office and rerun. Then write the twenty-question evaluation set that matters to
   your users, not to this notebook.
2. **Move the chunk size.** Halve it, double it, and record the retrieval hit rate
   each time. Put the three numbers in your Friday repository — that table is a
   methodological finding.
3. **Make the citation clickable.** Add a page number to `Chunk` when the source is
   a PDF, so an answer points at a page rather than a passage index.
4. **Retrieve twice.** Retrieve with TF-IDF and with embeddings, merge the results,
   and deduplicate. Measure whether the hit rate improves enough to justify running
   both.

Tomorrow at 15:45 this assistant becomes an agent: the model stops receiving
passages you chose and starts choosing which tool to call. Everything you learned
about refusal applies there too, with higher stakes — a wrong sentence becomes a
wrong action.


---
> ### If something did not work
>
> **No model provider.** Steps 1–4 and 8–9 run without one, and they are the
> retrieval half — where most RAG problems actually live. Set one API key before
> Day 2 and rerun steps 5–7.
>
> **The embedding download failed.** TF-IDF is a legitimate retriever, not a
> degraded mode. Record which one you used and carry on.
>
> **Your PDFs produced nothing.** They are scans. They need OCR, which is outside
> this laboratory. Use the sample corpus today and bring the question to Day 3.
>
> **Everything is refused.** `MIN_SCORE` is too high. Set it to `0.0` and rerun
> step 8 with more questions.
